# 一键流程：PDF 文件夹 → 入库 → 发布 → 回答问题集

从上到下依次执行即可：对一个文件夹里的每份 PDF 依次做 ingest → requalify → qualify → index → publish → tree，
然后（可选）用同一套 `document-catalog` 链路在进程内回答一份问题集，并写出 `report.json` / `report.md`。
已处理过的 PDF 重跑只回放缓存，不再调用模型。

## 前置条件

1. 项目根 `.env` 按 `.env.example` 填好（`.env` 固定在项目根，与 notebook 的工作目录无关，不提交）：
   - LLM：`OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`（别名 `APP_LLM_API_KEY` / `APP_LLM_BASE_URL` / `APP_LLM_MODEL`）；
   - Embedding：最简单是设 `OPENAI_EMBEDDING_MODEL`（与 LLM 同一个 `OPENAI_BASE_URL` 网关、同一个 key）；独立 loopback 服务才用 `APP_EMBEDDING_*`；
   - Rerank：`APP_RERANK_*`（本地模型，loopback 地址；没有网关回退）；
   - 输入输出位置：`NB_PDF_DIR`（必填）、`NB_QUESTIONS_PATH`（别名 `DATASET_PATH`）、`NB_REPORT_DIR`（后两项可选）。
2. 用到本地模型（独立 embedding 服务或 rerank）时，**本地模型隧道要先起**（本流程自己不会启动它）：
   `.venv/bin/python scripts/enterprise_pdf_rag/local_model_tunnel.py start`
3. 用项目的 `.venv` 作为 kernel（已 `pip install -e .`）。

本 notebook 不含任何密钥，也不向 `os.environ` 写入任何值。

## 测试数据

默认的合成 PDF 目录 `data/fixtures/pdf` 被 `.gitignore` 的 `data/*` 忽略，不随仓库同步；机器上没有时用以下命令生成（需要 dev 依赖 reportlab）：

```bash
.venv/bin/python scripts/make_fixtures_pdf.py
```

## 运行时行为

- 缺少必需配置时会在花任何 LLM 预算之前抛 `PreflightError` 并指出缺哪些变量。
- 开跑前会向 embedding 服务探测一次（`embed_query("preflight")`），用来确认网关或隧道已通。
- 只有题集里有 `rerank: true` 的用例时才需要 `APP_RERANK_*` 配置。

In [ ]:
# 安装本仓库（editable）及一键流程需要的 extras：pdf = pdfspine，service = 进程内评测用的 fastapi/httpx。
# 已经装过的环境可以跳过这一格。Databricks 上执行完要重启解释器：dbutils.library.restartPython()
%pip install -q -e "..[pdf,service]"

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
# 本格是"notebook 不直接读环境变量"规则的例外：它在 import ragspine 之前执行，拿不到 get_settings()，只能直接读环境变量做平台探测
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# editable 安装靠 .pth 把 <repo>/src 加进 sys.path；Databricks 上该 .pth 不生效时，
# site-packages/ragspine/ 只剩 _llms/（无 __init__.py），import ragspine 会命中这个
# namespace package，于是找不到 ragspine.common。这里显式把 src 放到 sys.path 最前
# （必须在 restart 之后）；.pth 已生效的环境里这只是无害的重复。
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if (_src / "ragspine" / "__init__.py").is_file() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
print("src 路径 =", _src, "| 已在 sys.path:", str(_src) in sys.path)

In [ ]:
# 导入诊断：确认 import ragspine 实际命中的是哪份代码（restart 之后执行）
# - __file__ 为 None            → 命中了同名目录形成的 namespace package，看 __path__ 是谁在遮蔽（见上一格 src-path）
# - __file__ 在 site-packages 下 → 装的是 PyPI 发行版（0.17.2 及更早没有 common/evidence/configs.py）
# - __file__ 指向 .../src/ragspine/__init__.py → 上面的 editable 安装已生效
import importlib.metadata as md
import importlib.util
import sys

try:
    import ragspine
except ImportError as exc:
    ragspine = None
    print("import ragspine 失败 =", repr(exc))

if ragspine is not None:
    print("ragspine.__file__ =", ragspine.__file__)
    print("ragspine.__path__ =", list(getattr(ragspine, "__path__", [])))
    for mod in ("ragspine.common", "ragspine.common.evidence", "ragspine.common.evidence.configs"):
        try:
            found = importlib.util.find_spec(mod) is not None
        except ImportError:
            found = False
        print(f"{mod}: {'OK' if found else '找不到'}")
for dist in ("ragspine", "rag-spine"):
    try:
        print(f"{dist} 版本 =", md.version(dist))
    except md.PackageNotFoundError:
        print(f"{dist} 未安装")
print("sys.path 前 8 项 =", sys.path[:8])

In [ ]:
from ragspine.common.evidence.configs import get_settings

settings = get_settings()

# ---- 来自项目根 .env（或真实环境变量）：未设置时为 None ----
PDF_DIR = settings.pdf_source_dir  # NB_PDF_DIR
QUESTIONS = settings.questions_path  # NB_QUESTIONS_PATH：不设则只入库、不评测
REPORT_DIR = settings.report_dir  # NB_REPORT_DIR：不设则不写报告
INGESTION_ROOT = settings.ingestion_root  # APP_INGESTION_DIR，默认 <APP_DATA_DIR>/ingestion

# ---- 可在这里覆盖的调用参数 ----
MAX_LIVE_CALLS_PER_PDF = 100  # 每份 PDF 的 ingest 真实模型调用预算（0-200）；0 = 只回放缓存
MAX_LIVE_CALLS_TOTAL = None  # ingest + tree + 回答共用的总预算；None = 不设总预算
BUILD_TREE = True  # 发布后为每份文档建 document tree（路由用，会多花调用）
REQUALIFY = True  # 重新资格化图表等视觉对象（图表要靠它才进索引）

print("PDF 源目录 :", PDF_DIR)
print("题集       :", QUESTIONS)
print("报告目录   :", REPORT_DIR)
print("ingestion  :", INGESTION_ROOT)

In [ ]:
from enterprise_pdf_rag.adapters.folder_pipeline import run_folder_pipeline


def progress(event, payload):
    print(event, payload)


result = run_folder_pipeline(
    PDF_DIR,  # None → 取 NB_PDF_DIR；两处都没有会直接报错
    questions=QUESTIONS,
    ingestion_root=INGESTION_ROOT,
    max_live_calls_per_pdf=MAX_LIVE_CALLS_PER_PDF,
    max_live_calls_total=MAX_LIVE_CALLS_TOTAL,
    build_tree=BUILD_TREE,
    requalify=REQUALIFY,
    report_dir=REPORT_DIR,
    progress=progress,
)

In [ ]:
from pathlib import Path


def show(rows, columns):
    """不依赖 pandas 的等宽表格。"""
    cells = [[str(row.get(column, "")) for column in columns] for row in rows]
    widths = [
        max([len(column), *(len(line[i]) for line in cells)]) for i, column in enumerate(columns)
    ]
    print("  ".join(column.ljust(width) for column, width in zip(columns, widths)))
    for line in cells:
        print("  ".join(value.ljust(width) for value, width in zip(line, widths)))


print("ok              :", result.ok)
print("live_calls      :", result.live_calls.model_dump())
print("budget_exhausted:", result.budget_exhausted)
print("report_dir      :", result.report_dir)

documents = [
    {
        "pdf": Path(item.pdf_path).name,
        "status": item.status,
        "live_calls": item.live_calls,
        "elapsed_s": round(item.elapsed_s, 1),
        # 图表是否进了索引：qualification.chart_member_count > 0
        "chart_member_count": None
        if item.qualification is None
        else item.qualification.chart_member_count,
        "indexed_members": None if item.publication is None else item.publication.member_count,
        "error": item.error or "",
    }
    for item in result.documents
]
show(
    documents,
    ["pdf", "status", "live_calls", "elapsed_s", "chart_member_count", "indexed_members", "error"],
)

In [ ]:
if result.eval is None:
    print("没有题集（NB_QUESTIONS_PATH 未设置）：只做了入库。")
else:
    print("totals:", result.eval.totals)
    show(
        [
            {
                "case": case.case_id,
                "verdict": case.verdict,
                "status": case.status or "",
                "claims": case.claim_count,
                "cited_pages": list(case.cited_pages),
                "page_rank": case.page_rank,
                "ms": round(case.elapsed_ms),
                "failures": "; ".join(case.failures)[:80],
            }
            for case in result.eval.cases
        ],
        ["case", "verdict", "status", "claims", "cited_pages", "page_rank", "ms", "failures"],
    )

## 可选：单独拿一个回答模型客户端

`make_answer_llm()` 用 `.env` 里的 LLM 配置（`OPENAI_*`，别名 `APP_LLM_*`）构造 `JsonCompletionClient`，
缓存目录、调用预算、超时与种子取自 settings。构造本身不发请求；配置缺失时在这里抛 `ProviderConfigurationError`。

In [ ]:
from enterprise_pdf_rag.adapters.answer_llm import make_answer_llm

llm = make_answer_llm()  # 或 make_answer_llm(cache_dir=..., max_live_calls=...)
print(type(llm).__name__, "live calls so far:", llm.live_call_count)